# Olist Data Statistics & Data Quality Audit
## Audit Objectives
- Check primary key uniqueness and duplicated review records;
- Verify logical order of order timestamps and filter invalid time records;
- Calculate distribution of delivery days & delay days, detect extreme outliers;
- Establish sample exclusion rules and count remaining valid orders after each filtering step.
### Tools
DuckDB(SQL for query & count) + Pandas(statistics & visualization).

In [1]:
from pathlib import Path
import pandas as pd
import duckdb
import numpy as np

project_root = Path.cwd()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
orders_path = raw_data_dir / "olist_orders_dataset.csv"
reviews_path = raw_data_dir / "olist_order_reviews_dataset.csv"

df_orders = pd.read_csv(orders_path, parse_dates=[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
df_reviews = pd.read_csv(reviews_path)

conn = duckdb.connect()
conn.register("orders", df_orders)
conn.register("reviews", df_reviews)

total_orders = df_orders.shape[0]
print(f"Total original orders: {total_orders}")
print("Data folder path:", raw_data_dir.resolve())

Total original orders: 99441
Data folder path: C:\Users\H\Desktop\folder\ecommerce-delivery-analysis\data\raw


## Primary Key Uniqueness Check
### Check items
- Whether  order_id  in orders table is globally unique;
- Duplicated  review_id  and multiple reviews for one single order in reviews table.
- Impact:Duplicated reviews will cause biased rating statistics and incorrect sample size.

In [2]:
sql1 = """
SELECT order_id, COUNT(*) as cnt
FROM orders
GROUP BY order_id
HAVING cnt>1
"""
dup_order = conn.execute(sql1).fetchdf()
print("Count of duplicated order_id: ", len(dup_order))

sql2 = """
SELECT review_id, COUNT(*) cnt FROM reviews GROUP BY review_id HAVING cnt>1;
"""
dup_review_id = conn.execute(sql2).fetchdf()

sql3 = """
SELECT order_id, COUNT(*) review_cnt
FROM reviews
GROUP BY order_id
HAVING review_cnt>1
ORDER BY review_cnt DESC
"""
multi_review_order = conn.execute(sql3).fetchdf()
print("Count of orders with multiple reviews: ", len(multi_review_order))
print("Distribution of review counts per order:")
print(multi_review_order["review_cnt"].value_counts().sort_index())

Count of duplicated order_id:  0
Count of orders with multiple reviews:  547
Distribution of review counts per order:
review_cnt
2    543
3      4
Name: count, dtype: int64


## Suggestions for handling duplicated reviews (to be confirmed later)
- Keep the latest review (max review creation date);
- Calculate average score for multiple reviews under one order;
- Conservatively drop all orders with multiple reviews.

## Validation of Timestamp Logic
Normal time sequence:
Purchase time → Order approved time → Hand over to carrier → Customer delivered time
Filter rules: missing key dates, reversed time sequence, delivery time earlier than purchase time.
Invalid time data will lead to negative delivery days and wrong statistical results.

In [3]:
sql_time = """
SELECT *,
order_purchase_timestamp,
order_approved_at,
order_delivered_carrier_date,
order_delivered_customer_date
FROM orders
WHERE
-- Key delivery date is null
order_delivered_customer_date IS NULL
OR order_purchase_timestamp IS NULL
-- Time sequence reversed
OR order_approved_at < order_purchase_timestamp
OR order_delivered_carrier_date < order_approved_at
OR order_delivered_customer_date < order_delivered_carrier_date
"""
error_time_df = conn.execute(sql_time).fetchdf()
print(f"Count of orders with abnormal time / missing dates: {len(error_time_df)}")
error_order_ids = error_time_df["order_id"].unique()

Count of orders with abnormal time / missing dates: 4338


## Calculate Delivery & Delay Days + Descriptive Statistics
### Indicator Definition
- Delivery days = delivered date - purchase date (days)
- Delay days = delivered date - estimated delivery date (positive value = delayed)
Metrics: min, mean, median, 95th percentile, max

In [4]:
df_valid = df_orders.dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"]
).copy()

df_valid["delivery_days"] = (
    df_valid["order_delivered_customer_date"] - df_valid["order_purchase_timestamp"]
).dt.days
df_valid["delay_days"] = (
    df_valid["order_delivered_customer_date"] - df_valid["order_estimated_delivery_date"]
).dt.days

stat_cols = ["delivery_days", "delay_days"]
desc_stats = df_valid[stat_cols].describe(percentiles=[0.05, 0.5, 0.95])
print("Statistics of delivery days & delay days:")
desc_stats

Statistics of delivery days & delay days:


,delivery_days,delay_days
count,96476.000000,96476.000000
mean,12.094086,-11.876881
std,9.551746,10.183854
min,0.000000,-147.000000
5%,3.000000,-26.000000
50%,10.000000,-12.000000
95%,29.000000,3.000000
max,209.000000,188.000000


## Outlier Judgment & Preliminary Sample Exclusion Rules 
### Judgment logic
- Negative delivery days, reversed time, missing key dates → Directly removed (data input error);
- Extreme delay far above 95% percentile: distinguish real logistics delay or wrong timestamp, confirm with business logic later;
- Orders with duplicated reviews: 3 optional solutions for team discussion.

In [5]:
import pandas as pd
import numpy as np
from IPython.display import display

raw_total = len(df_orders)

df_step1 = df_orders[~df_orders["order_id"].isin(error_order_ids)].copy()
step1_drop = raw_total - len(df_step1)
step1_left = len(df_step1)

df_step2 = df_step1[df_step1["order_status"]=="delivered"].copy()
step2_drop = step1_left - len(df_step2)
step2_left = len(df_step2)

df_step2 = df_step2.assign(
    delay_days = lambda x: (x["order_delivered_customer_date"] - x["order_estimated_delivery_date"]).dt.days
)

extreme_delay_count = len(df_step2[df_step2["delay_days"] > 60])
final_df = df_step2[df_step2["delay_days"] <= 60].copy()
final_count = len(final_df)

summary = pd.DataFrame({
    "Stage": ["Raw Data", "Remove Abnormal Time", "Only Delivered Orders", "Remove Extreme Delay"],
    "Remaining Orders": [raw_total, step1_left, step2_left, final_count],
    "Dropped In This Step": [0, step1_drop, step2_drop, extreme_delay_count]
})

print("=== Sample Filtering Summary Table ===")
display(summary)

=== Sample Filtering Summary Table ===


,Stage,Remaining Orders,Dropped In This Step
0,Raw Data,99441,0
1,Remove Abnormal Time,95103,4338
2,Only Delivered Orders,95097,6
3,Remove Extreme Delay,95018,79


## Conclusion & Items to be discussed together
- Which rule to apply for duplicated reviews?
-  Threshold for removing extreme delayed orders (whether 60 days is proper)
-  Whether to keep advanced-delivery records with negative delay days?
Subsequent modeling will only use samples filtered by the above rules.